# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [ ]:
team_setup = {
    'team_name': 'The Terrific Trio',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Ethan', 'github': 'ethan-y-cheung'},
        {'name': 'Steve', 'github': 'sphilipose6'},
        {'name': 'Henry', 'github': ''},
    ],
    'repo_url': 'https://github.com/sphilipose6/dss2002_Midterm',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/sphilipose6/dss2002_Midterm/blob/main/CHARTER.md',  # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Henry',          # nice to have
    'meeting_time': 'Sun and Fri 7pm',            # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': [                      # nice to have, five specific problems
        "store_id has 60 distinct values for 12 stores: FL-239, FL_239, FL239, fl-239 and ' FL-239' with a leading space",
        "timestamp mixes 5 formats (2024-09-24 09:48:13, 2024-09-24T09:48:13, 09/26/2024 09:26, 09-23-2024 18:24:47, 21-Sep-2024 19:46) and 886 are blank, NULL, N/A, NaN or nan",
        "unit_price is stored as text and some values start with $ or a space",
        "45 SKUs but only 20 real products, e.g. strawberry Pop-Tarts is PT-12, SKU#459812, POPTART-STRAW and PT_STRAWBERRY",
        "category has 37 spellings of 10 categories (EMERGENCY, Emergency Supplies, emergency...), plus 130 negative quantities and 30 duplicate rows",
    ],
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [ ]:
import subprocess

REPO_PATH = '..'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/'
NA = ['', 'NULL', 'N/A', 'NaN', 'nan']

decisions = []
def log(step, decision, rows):
    decisions.append({'step': step, 'decision': decision, 'rows_touched': int(rows)})


def clean_store_id(s):
    s = s.astype(str).str.strip().str.upper().str.replace('_', '-')
    return s.str.replace(r'^FL(\d{3})$', r'FL-\1', regex=True)

def parse_timestamps(s):
    formats = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M', '%m-%d-%Y %H:%M:%S', '%d-%b-%Y %H:%M']
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for f in formats:
        out = out.combine_first(pd.to_datetime(s, format=f, errors='coerce'))
    return out

CATEGORY_MAP = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks',
    'health': 'Health', 'health & wellness': 'Health',
}

def clean_category(s):
    return s.str.strip().str.lower().map(CATEGORY_MAP)


def clean_transactions(raw):
    df = raw.copy()

    n = len(df)
    df = df.drop_duplicates()
    log('duplicates', 'dropped exact duplicate rows (same transaction_id and every other column)', n - len(df))
    assert df['transaction_id'].is_unique

    new = clean_store_id(df['store_id'])
    log('store_id', 'strip, uppercase, _ -> -, FL239 -> FL-239', (new != df['store_id']).sum())
    df['store_id'] = new

    price = df['unit_price'].astype(str).str.strip().str.lstrip('$')
    log('unit_price', 'removed $ and spaces, converted text to float', (price != df['unit_price']).sum())
    df['unit_price'] = pd.to_numeric(price, errors='raise')

    df['quantity'] = df['quantity'].astype(int)
    neg = df['quantity'] < 0
    log('quantity', 'dropped negative quantities (returns, not demand)', neg.sum())
    df = df[~neg]

    df['timestamp'] = parse_timestamps(df['timestamp'])
    missing = df['timestamp'].isna()
    log('timestamp', 'parsed 5 formats as local store time; dropped rows with no timestamp (blank/NULL/N/A/NaN)', missing.sum())
    df = df[~missing]

    cat = clean_category(df['category'])
    assert cat.notna().all()
    log('category', 'mapped 37 spellings onto 10 categories (names match the inventory db)', (cat != df['category']).sum())
    df['category'] = cat

    df['product_name'] = df['product_name'].str.strip()
    df['revenue'] = df['quantity'] * df['unit_price']
    return df.reset_index(drop=True)


def clean_stores(raw):
    df = raw.copy()
    df['store_id'] = clean_store_id(df['store_id'])
    n = len(df)
    df = df.drop_duplicates('store_id')
    log('stores', 'dropped duplicate store rows after fixing store_id (FL239, FL_287)', n - len(df))
    log('stores', 'state Florida / Fla. -> FL, stripped region', (df['state'] != 'FL').sum())
    df['state'] = 'FL'
    df['region'] = df['region'].str.strip()
    tlh = df['latitude'].isna()
    df.loc[df['store_id'] == 'FL-619', ['latitude', 'longitude']] = [30.4383, -84.2807]
    log('stores', 'filled missing lat/lon for FL-619 Tallahassee with city center coordinates', tlh.sum())
    df['tz'] = np.where(df['store_id'] == 'FL-710', 'America/Chicago', 'America/New_York')
    return df.reset_index(drop=True)


raw_tx = pd.read_csv(DATA + 'walmart_transactions.csv', dtype=str, keep_default_na=False)
raw_stores = pd.read_csv(DATA + 'store_locations.csv', na_values=NA)

tx = clean_transactions(raw_tx)
stores = clean_stores(raw_stores)

print('transactions:', len(raw_tx), '->', len(tx), f'({len(raw_tx) - len(tx)} removed)')
print('stores:', len(raw_stores), '->', len(stores))
print('store ids in tx:', tx['store_id'].nunique(), '| all join to store file:', tx['store_id'].isin(stores['store_id']).all())
print('date range:', tx['timestamp'].min(), 'to', tx['timestamp'].max())
tx.head()

### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [ ]:
# TODO: print or display your decision log
pd.set_option('display.max_colwidth', None)
pd.DataFrame(decisions)

### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [ ]:
import re

catalog = pd.read_csv(DATA + 'product_catalog.csv', na_values=NA)

PRODUCT_KEYWORDS = [
    ('pop-?tart', 'Pop-Tarts Strawberry'),
    ('water', 'Bottled Water 24-Pack'),
    ('batter', 'AA Batteries 4-Pack'),
    ('flashlight', 'LED Flashlight'),
    ('beer', 'Light Beer 24-Pack'),
    ('monopoly', 'Monopoly Board Game'),
    ('playing cards', 'Playing Cards'),
    ('roku|streaming', 'Roku Streaming Stick'),
    ('soup', 'Canned Soup'),
    ('bread', 'White Bread'),
    ('peanut', 'Peanut Butter'),
    ('candle', 'Emergency Candles'),
    ('first.aid', 'First Aid Kit'),
    ('charger', 'Portable Phone Charger'),
    ('chips', 'Chips Variety Pack'),
    ('paper towel', 'Paper Towels'),
    ('toilet', 'Toilet Paper 12-Roll'),
    ('diaper', 'Baby Diapers Size 3'),
    ('formula', 'Baby Formula'),
    ('tarp', 'Heavy Duty Tarp'),
]

def canonical_product(name):
    name = name.lower()
    hits = [p for kw, p in PRODUCT_KEYWORDS if re.search(kw, name)]
    assert len(hits) == 1, (name, hits)
    return hits[0]

def build_sku_map(catalog):
    m = catalog[['sku', 'product_name', 'category']].copy()
    m['product'] = m['product_name'].apply(canonical_product)
    m['category'] = clean_category(m['category'])
    return m

sku_map = build_sku_map(catalog)

print('categories per product:', sku_map.groupby('product')['category'].nunique().max())

before = tx['sku'].nunique()
tx['product'] = tx['sku'].map(sku_map.set_index('sku')['product'])
assert tx['product'].notna().all()

wrong_cat = (tx['category'] != tx['sku'].map(sku_map.set_index('sku')['category'])).sum()
log('sku', 'mapped 45 SKUs to 20 canonical products by keyword on the catalog name', len(tx))
print('transaction rows whose category disagrees with the catalog:', wrong_cat)

groups = sku_map.groupby('product')['sku'].apply(list)
print('distinct SKUs:', before, '-> canonical products:', tx['product'].nunique())
print('largest groups:', groups[groups.str.len() == groups.str.len().max()].to_dict())

sku_map[sku_map['product'] == 'Pop-Tarts Strawberry']

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [ ]:
import requests
import time

BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'

def get_with_retries(params, retries=4):
    for attempt in range(retries):
        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            print('got status', r.status_code, 'retrying')
        except requests.RequestException as e:
            print('request failed:', e)
        time.sleep(2 ** attempt)
    raise RuntimeError(f'weather API failed after {retries} tries')


def fetch_weather(store, start='2024-09-01', end='2024-09-30'):
    params = {
        'latitude': store['latitude'], 'longitude': store['longitude'],
        'start_date': str((pd.Timestamp(start) - pd.Timedelta(days=1)).date()),
        'end_date': str((pd.Timestamp(end) + pd.Timedelta(days=1)).date()),
        'hourly': 'temperature_2m,precipitation,wind_speed_10m,wind_gusts_10m,pressure_msl',
        'temperature_unit': 'fahrenheit', 'wind_speed_unit': 'mph', 'precipitation_unit': 'inch',
    }
    data = get_with_retries(params)
    print('API timezone:', data['timezone'], '| utc offset:', data['utc_offset_seconds'])

    hourly = pd.DataFrame(data['hourly'])
    hourly['time_utc'] = pd.to_datetime(hourly['time']).dt.tz_localize('UTC')
    hourly['time_local'] = hourly['time_utc'].dt.tz_convert(store['tz'])
    hourly['date'] = hourly['time_local'].dt.date
    hourly = hourly[(hourly['date'] >= pd.Timestamp(start).date()) & (hourly['date'] <= pd.Timestamp(end).date())]

    daily = hourly.groupby('date').agg(
        hours=('time', 'size'),
        max_temp_f=('temperature_2m', 'max'),
        precip_in=('precipitation', 'sum'),
        max_wind_mph=('wind_speed_10m', 'max'),
        max_gust_mph=('wind_gusts_10m', 'max'),
        min_pressure_hpa=('pressure_msl', 'min'),
    ).reset_index()
    daily.insert(0, 'store_id', store['store_id'])
    daily['source'] = 'open-meteo: ' + BASE_URL
    return hourly, daily


orlando = stores.set_index('store_id').loc['FL-239'].copy()
orlando['store_id'] = 'FL-239'
weather_hourly, weather = fetch_weather(orlando)

print('hourly rows returned:', len(weather_hourly), '| days:', len(weather), '| every day has 24 hours:', (weather['hours'] == 24).all())
weather.head()

In [ ]:
tx['date'] = tx['timestamp'].dt.date
orl_sales = (tx[tx['store_id'] == 'FL-239'].groupby('date')
             .agg(transactions=('transaction_id', 'size'), units=('quantity', 'sum')).reset_index())

joined = orl_sales.merge(weather, on='date', how='inner', validate='one_to_one')
print('sales days:', len(orl_sales), '| weather days:', len(weather), '| joined:', len(joined))
assert len(joined) == len(orl_sales) == len(weather)
joined[['date', 'transactions', 'units', 'max_wind_mph', 'max_gust_mph', 'precip_in', 'min_pressure_hpa']].tail(8)

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [ ]:
import matplotlib.pyplot as plt

daily = tx.groupby('date').size()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pd.to_datetime(daily.index), daily.values, marker='o', label='transactions per day (all 12 stores)')
ax.axvline(pd.Timestamp('2024-09-26'), color='red', linestyle='--', label='Helene landfall (Sep 26)')
ax.set_title('Daily order volume, September 2024')
ax.set_xlabel('Date')
ax.set_ylabel('Transactions')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(daily.describe().round(1))

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [ ]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': len(tx),       # an integer
    'decisions_logged': len(decisions),       # how many entries in your log
    'canonical_products': tx['product'].nunique(),       # distinct products after consolidation
    'weather_api_rows': len(weather_hourly),       # rows returned by your API call
    'notebook_runs_clean': True,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** the weather join for Q3. Open-Meteo returns GMT while sales are in local store time, and Pensacola (FL-710) is on Central time, not Eastern. So far we have fetched one store (Orlando). If the other 11 stores get joined on the GMT date, evening sales land on the wrong day and the lag analysis is off by one around landfall, which is exactly the window Q3 is about.

**Plan, with an owner and a date:** Steve runs `fetch_weather` for all 12 stores (it already converts to each store's `tz`) and builds the store x day weather table with the same row-count check as the Orlando join (12 stores x 30 days = 360 rows) by Wednesday, Oct 28. Ethan is the backup and reviews the join before Henry uses it for Q3.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | in progress: dates and categories clean, daily volume chart done, baseline window not picked yet | Henry |
| Q2 SKU consolidation | in progress: mapping table done (45 SKUs to 20 products), before/after comparison not started | Ethan |
| Q3 weather correlation | in progress: API call and checked join for Orlando, other stores and correlation not started | Steve |
| Q4 store and geography | not started (store ids cleaned, all 12 join to the store file) | Henry |
| Q5 signal or artifact | not started | Ethan |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.